In [1]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
import torch
import torch_geometric
import scipy.sparse as sp
from torch_geometric.data import Data
from tqdm import tqdm

/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/torch_geometric/typing.py:54: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.29' not found (required by /gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/torch_geometric/typing.py:110: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /lib64/libm.so.6: version `GLIBC_2.29' not found (required by /gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/libpyg.so)
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "


In [42]:
import torch
import torch.nn.functional as F
from torch_scatter import scatter
from torch_geometric.nn import GCNConv
from torch.nn import Linear
from torch_geometric.data import Data
import torch
from torch_geometric.data import Data, DataLoader

class StatisticalPooling(torch.nn.Module):
    def __init__(self):
        super(StatisticalPooling, self).__init__()

    def forward(self, x, batch):
        # Ensure 'batch' is not None and is properly sized
        if batch is None:
            raise ValueError("Batch tensor must not be None")

        # Mean
        mean = scatter(x, batch, dim=0, reduce='mean')

        # # Variance
        mean_sq = scatter(x**2, batch, dim=0, reduce='mean')
        variance = mean_sq - mean**2

        # # Correct the indexing for mean and variance when used
        mean_expanded = mean[batch]
        # variance_expanded = variance[batch]

        # # Skewness
        n = scatter(torch.ones_like(x[:, 0]), batch, dim=0, reduce='sum').unsqueeze(-1)  # number of elements per batch
        skewness = scatter((x - mean_expanded)**3, batch, dim=0, reduce='sum') / (n * (variance**1.5 + 1e-6))

        # # Kurtosis
        kurtosis = scatter((x - mean_expanded)**4, batch, dim=0, reduce='sum') / (n * (variance**2 + 1e-6)) - 3

        # # Concatenate all statistics to form a single vector per graph
        pooled_features = torch.cat([mean, variance, skewness, kurtosis], dim=1)
        # pooled_features = torch.cat([mean, variance, skewness], dim=1)
        return pooled_features



class GNNModel(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(GNNModel, self).__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.pool = StatisticalPooling()
        self.fc = Linear(hidden_dim * 4, output_dim)  # Adjust for concatenated stats

    def reset_parameters(self):
        # Set the weights of conv1 to ones
        self.conv1.reset_parameters()  # Resets the parameters to default initialization

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = self.conv1(x, edge_index)
        # x = F.relu(x)
        x = self.pool(x, batch)  # Ensure batch is correctly passed
        # x = self.fc(x)
        return x

# Dummy Data initialization and model usage can continue here


In [43]:
data = torch.load('../data/hgsae2/embedding_graph_dataset.pt')

In [44]:
data

[Data(x=[8, 32], edge_index=[2, 38], edge_attr=[38, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 38], edge_attr=[38, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 38], edge_attr=[38, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 36], edge_attr=[36, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 36], edge_attr=[36, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 32], edge_attr=[32, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 40], edge_attr=[40, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 38], edge_attr=[38, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 32], edge_attr=[32, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 40], edge_attr=[40, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 32], edge_attr=[32, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 40], edge_attr=[40, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 36], edge_attr=[36, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 38], edge_attr=[38, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 40], edge_attr=[40, 1], y=[1]),
 Data(x=[8, 32], edge_index=[2, 36], edg

In [45]:
gnn = GNNModel(input_dim=data[0].x.size(1), hidden_dim=16, output_dim=6)
gnn.reset_parameters()

In [46]:
loader = DataLoader(data, batch_size=53, shuffle=True)

/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/torch_geometric/deprecation.py:26: UserWarning:

'data.DataLoader' is deprecated, use 'loader.DataLoader' instead



In [47]:
b = next(iter(loader))

In [48]:
x_stats = gnn(b).detach().cpu().numpy()

In [49]:
b.y

tensor([4, 3, 1, 3, 0, 3, 3, 1, 3, 3, 1, 5, 2, 4, 3, 3, 3, 3, 3, 2, 2, 2, 2, 3,
        4, 3, 0, 3, 3, 4, 3, 3, 3, 2, 4, 4, 3, 4, 4, 3, 4, 3, 4, 2, 3, 5, 3, 3,
        2, 3, 3, 3, 4])

In [50]:
import phate
phate_op = phate.PHATE(n_components=2, random_state=4)
x_phate = phate_op.fit_transform(x_stats)

Calculating PHATE...
  Running PHATE on 53 observations and 64 variables.
  Calculating graph and diffusion operator...
    Calculating KNN search...
    Calculating affinities...
  Calculating optimal t...
    Automatically selected t = 20
  Calculating diffusion potential...
  Calculating metric MDS...
Calculated PHATE in 0.02 seconds.


/gpfs/gibbs/pi/krishnaswamy_smita/xingzhi/.conda_envs/geosink/lib/python3.11/site-packages/graphtools/graphs.py:283: RuntimeWarning:

Detected zero distance between samples 5 and 18. Consider removing duplicates to avoid errors in downstream processing.



In [51]:
from plotly3d.plot import scatter

In [53]:
scatter(x_phate,b.y,  s=5)